In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import numpy as np
import tensorflow as tf
import os
import json
import gc

from sklearn.metrics import f1_score, mean_absolute_error
from sklearn.model_selection import train_test_split

In [ ]:
MODEL_PATH = "/content/drive/MyDrive/FYP/models"
DATA_PATH = "/content/drive/MyDrive/FYP/phase3"

train_houses = list(range(1, 17))
test_houses = list(range(17, 21))

os.makedirs(MODEL_PATH, exist_ok=True)

MAX_SAMPLES = 8000

In [ ]:
def load_seq2point_predictions(app, X):
    model = tf.keras.models.load_model(
        f"{MODEL_PATH}/{app}_seq2point_final.h5",
        compile=False
    )
    return model.predict(X, verbose=0).flatten()


def load_data(app, houses):
    X, y, seq = [], [], []

    for house in houses:
        path = f"{DATA_PATH}/house{house}_{app}_seq2point.npz"
        if not os.path.exists(path):
            continue

        data = np.load(path)
        X_tmp = data['X']
        y_tmp = data['y']

        seq_tmp = load_seq2point_predictions(app, X_tmp)

        if len(X_tmp) > MAX_SAMPLES:
            idx = np.random.choice(len(X_tmp), MAX_SAMPLES, replace=False)
            X_tmp = X_tmp[idx]
            y_tmp = y_tmp[idx]
            seq_tmp = seq_tmp[idx]

        X.append(X_tmp)
        y.append(y_tmp)
        seq.append(seq_tmp)

    return np.concatenate(X), np.concatenate(y), np.concatenate(seq)

In [ ]:
def build_model(input_length, channels):
    model = tf.keras.Sequential([
        tf.keras.layers.Conv1D(64, 5, activation='relu', input_shape=(input_length, channels)),
        tf.keras.layers.MaxPooling1D(2),

        tf.keras.layers.Conv1D(128, 3, activation='relu'),
        tf.keras.layers.MaxPooling1D(2),

        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.3),

        tf.keras.layers.Dense(1, activation='sigmoid')
    ])

    return model

In [ ]:
def focal_loss(alpha=0.75, gamma=2.0):
    def loss(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
        pt = tf.where(tf.equal(y_true, 1), y_pred, 1 - y_pred)
        return -tf.reduce_mean(alpha * tf.pow(1 - pt, gamma) * tf.math.log(pt))
    return loss

In [ ]:
def train_classifier(APP):

    print("\n===== TRAINING:", APP, "=====")

    # =========================
    # SPECIAL CASE: KETTLE
    # =========================
    if APP == "kettle":

        print("Using STABLE kettle mode")

        X_train, y_train, _ = load_data(APP, train_houses)
        X_test, y_test, _ = load_data(APP, test_houses)

        X_train = X_train[..., np.newaxis]
        X_test = X_test[..., np.newaxis]

        print("y_train min:", np.min(y_train))
        print("y_train max:", np.max(y_train))

        # ORIGINAL WORKING THRESHOLD
        threshold = np.max(y_train) * 0.1

        y_train_bin = (y_train > threshold).astype(int)
        y_test_bin = (y_test > threshold).astype(int)

        print("Threshold:", threshold)
        print("ON ratio:", np.mean(y_train_bin))

        # ✅ SAFE CLASS WEIGHT
        on_ratio = np.mean(y_train_bin)
        class_weights = {
            0: 1.0,
            1: min(8, 1 / (on_ratio + 1e-6))
        }

        model = build_model(X_train.shape[1], 1)

        # ✅ IMPORTANT: NORMAL LOSS (NOT focal)
        model.compile(
            optimizer='adam',
            loss='binary_crossentropy',
            metrics=['accuracy']
        )

        model.fit(
            X_train, y_train_bin,
            epochs=100,
            batch_size=64,
            class_weight=class_weights,
            verbose=1
        )

        # PREDICTION
        y_pred_prob = model.predict(X_test).flatten()

        # THRESHOLD TUNING
        best_f1 = 0
        best_th = 0.5

        for th in np.linspace(0.3, 0.7, 15):
            y_pred_bin = (y_pred_prob > th).astype(int)
            f1_temp = f1_score(y_test_bin, y_pred_bin, zero_division=0)

            if f1_temp > best_f1:
                best_f1 = f1_temp
                best_th = th

        print("Best Threshold:", best_th)

        y_pred_bin = (y_pred_prob > best_th).astype(int)

        f1 = f1_score(y_test_bin, y_pred_bin, zero_division=0)
        mae = mean_absolute_error(y_test, y_pred_prob)

        print("FINAL F1:", f1)
        print("MAE:", mae)

        model.save(f"{MODEL_PATH}/{APP}_classifier.h5")

        del model
        gc.collect()

        return {"F1": float(f1), "MAE": float(mae)}

    # =========================
    #  NORMAL APPLIANCES
    # =========================
    X_train, y_train, seq_train = load_data(APP, train_houses)
    X_test, y_test, seq_test = load_data(APP, test_houses)

    X_train = X_train[..., np.newaxis]
    X_test = X_test[..., np.newaxis]

    # USE SEQ2POINT FOR OTHERS ONLY
    seq_mean = np.mean(seq_train)
    seq_std = np.std(seq_train) + 1e-6

    seq_train = (seq_train - seq_mean) / seq_std
    seq_test = (seq_test - seq_mean) / seq_std

    seq_train = np.repeat(seq_train[:, np.newaxis], X_train.shape[1], axis=1)
    seq_test = np.repeat(seq_test[:, np.newaxis], X_test.shape[1], axis=1)

    seq_train = seq_train[..., np.newaxis]
    seq_test = seq_test[..., np.newaxis]

    X_train = np.concatenate([X_train, seq_train], axis=-1)
    X_test = np.concatenate([X_test, seq_test], axis=-1)

    # =========================
    # LABELS
    # =========================
    threshold = np.percentile(y_train, 50)

    y_train_bin = (y_train > threshold).astype(int)
    y_test_bin = (y_test > threshold).astype(int)

    print("Threshold:", threshold)
    print("ON ratio:", np.mean(y_train_bin))

    # =========================
    # SPLIT
    # =========================
    X_train, X_val, y_train_bin, y_val_bin = train_test_split(
        X_train, y_train_bin, test_size=0.2, random_state=42
    )

    # =========================
    # CLASS WEIGHTS
    # =========================
    classes = np.unique(y_train_bin)

    if len(classes) < 2:
        class_weights = None
    else:
        weights = compute_class_weight('balanced', classes=classes, y=y_train_bin)
        class_weights = {int(c): w for c, w in zip(classes, weights)}

    # =========================
    # MODEL
    # =========================
    model = build_model(X_train.shape[1], X_train.shape[2])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-4),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    model.fit(
        X_train, y_train_bin,
        validation_data=(X_val, y_val_bin),
        epochs=100,
        batch_size=64,
        class_weight=class_weights,
        verbose=1
    )

    # =========================
    # THRESHOLD TUNING
    # =========================
    y_val_prob = model.predict(X_val).flatten()

    best_f1 = 0
    best_th = 0.5

    for th in np.linspace(0.2, 0.7, 30):
        pred = (y_val_prob > th).astype(int)
        f1 = f1_score(y_val_bin, pred, zero_division=0)

        if f1 > best_f1:
            best_f1 = f1
            best_th = th

    print("Best Threshold:", best_th)

    # =========================
    # TEST
    # =========================
    y_pred = model.predict(X_test).flatten()
    y_pred_bin = (y_pred > best_th).astype(int)

    f1 = f1_score(y_test_bin, y_pred_bin, zero_division=0)
    mae = mean_absolute_error(y_test, y_pred)

    print("FINAL F1:", f1)
    print("MAE:", mae)

    model.save(f"{MODEL_PATH}/{APP}_classifier.h5")

    del model
    gc.collect()

    return {"F1": float(f1), "MAE": float(mae)}

In [ ]:
APP = "kettle"

result_kettle = train_classifier(APP)
print(result_kettle)


===== TRAINING: kettle =====
Using STABLE kettle mode
y_train min: 0.0
y_train max: 1.0106666
Threshold: 0.101066664
ON ratio: 0.006614583333333333


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/100
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 16s 9ms/step - accuracy: 0.9927 - loss: 0.1097
Epoch 2/100
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 13s 9ms/step - accuracy: 0.9910 - loss: 0.0742
Epoch 3/100
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 13s 9ms/step - accuracy: 0.9894 - loss: 0.0675
Epoch 4/100
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 13s 9ms/step - accuracy: 0.9902 - loss: 0.0571
Epoch 5/100
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 13s 9ms/step - accuracy: 0.9894 - loss: 0.0552
Epoch 6/100
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - accuracy: 0.9897 - loss: 0.0494
Epoch 7/100
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 13s 9ms/step - accuracy: 0.9898 - loss: 0.0507
Epoch 8/100
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 13s 9ms/step - accuracy: 0.9908 - loss: 0.0412
Epoch 9/100
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 13s 9ms/step - accuracy: 0.9918 - loss: 0.0373
Epoch 10/100
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 13s 9ms/step - accuracy: 0.9927 - loss: 0.0322
Epoch 11/100
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 13s 9ms/step - accuracy: 0.9933 - loss: 0.0298
Epoch 12

Best Threshold: 0.6714285714285715
FINAL F1: 0.75
MAE: 0.003658831585198641
{'F1': 0.75, 'MAE': 0.003658831585198641}


In [ ]:
APP = "toaster"

result_toaster = train_classifier(APP)
print(result_toaster)


===== TRAINING: toaster =====
Threshold: 0.0
ON ratio: 0.07013888888888889


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/100
900/900 ━━━━━━━━━━━━━━━━━━━━ 13s 10ms/step - accuracy: 0.5814 - loss: 0.5750 - val_accuracy: 0.5597 - val_loss: 0.5904
Epoch 2/100
900/900 ━━━━━━━━━━━━━━━━━━━━ 9s 10ms/step - accuracy: 0.5924 - loss: 0.5197 - val_accuracy: 0.6093 - val_loss: 0.5217
Epoch 3/100
900/900 ━━━━━━━━━━━━━━━━━━━━ 8s 9ms/step - accuracy: 0.6118 - loss: 0.4988 - val_accuracy: 0.5457 - val_loss: 0.6204
Epoch 4/100
900/900 ━━━━━━━━━━━━━━━━━━━━ 8s 9ms/step - accuracy: 0.6428 - loss: 0.4798 - val_accuracy: 0.6586 - val_loss: 0.5013
Epoch 5/100
900/900 ━━━━━━━━━━━━━━━━━━━━ 9s 10ms/step - accuracy: 0.6882 - loss: 0.4679 - val_accuracy: 0.7606 - val_loss: 0.4203
Epoch 6/100
900/900 ━━━━━━━━━━━━━━━━━━━━ 8s 9ms/step - accuracy: 0.7160 - loss: 0.4539 - val_accuracy: 0.7057 - val_loss: 0.5292
Epoch 7/100
900/900 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step - accuracy: 0.7326 - loss: 0.4416 - val_accuracy: 0.6808 - val_loss: 0.5409
Epoch 8/100
900/900 ━━━━━━━━━━━━━━━━━━━━ 9s 10ms/step - accuracy: 0.7424 - loss: 0.4336 - val

FINAL F1: 0.007692307692307693
MAE: 0.06391488760709763
{'F1': 0.007692307692307693, 'MAE': 0.06391488760709763}


In [ ]:
APP = "computer"

result_computer = train_classifier(APP)
print(result_computer)


===== TRAINING: computer =====
Threshold: 0.007
ON ratio: 0.499


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/100
900/900 ━━━━━━━━━━━━━━━━━━━━ 13s 10ms/step - accuracy: 0.7065 - loss: 0.5711 - val_accuracy: 0.7688 - val_loss: 0.4992
Epoch 2/100
900/900 ━━━━━━━━━━━━━━━━━━━━ 9s 10ms/step - accuracy: 0.7671 - loss: 0.4989 - val_accuracy: 0.7710 - val_loss: 0.4906
Epoch 3/100
900/900 ━━━━━━━━━━━━━━━━━━━━ 10s 11ms/step - accuracy: 0.7755 - loss: 0.4864 - val_accuracy: 0.7744 - val_loss: 0.4856
Epoch 4/100
900/900 ━━━━━━━━━━━━━━━━━━━━ 9s 10ms/step - accuracy: 0.7831 - loss: 0.4755 - val_accuracy: 0.7806 - val_loss: 0.4775
Epoch 5/100
900/900 ━━━━━━━━━━━━━━━━━━━━ 9s 10ms/step - accuracy: 0.7871 - loss: 0.4680 - val_accuracy: 0.7897 - val_loss: 0.4650
Epoch 6/100
900/900 ━━━━━━━━━━━━━━━━━━━━ 9s 10ms/step - accuracy: 0.7926 - loss: 0.4595 - val_accuracy: 0.7869 - val_loss: 0.4638
Epoch 7/100
900/900 ━━━━━━━━━━━━━━━━━━━━ 9s 10ms/step - accuracy: 0.7978 - loss: 0.4529 - val_accuracy: 0.7988 - val_loss: 0.4551
Epoch 8/100
900/900 ━━━━━━━━━━━━━━━━━━━━ 9s 10ms/step - accuracy: 0.8001 - loss: 0.4490 

FINAL F1: 0.6706615079739549
MAE: 0.45972582697868347
{'F1': 0.6706615079739549, 'MAE': 0.45972582697868347}


In [ ]:
APP = "lamp"

result_lamp = train_classifier(APP)
print(result_lamp)


===== TRAINING: lamp =====
Threshold: 0.0
ON ratio: 0.4180390625


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/100
1600/1600 ━━━━━━━━━━━━━━━━━━━━ 19s 10ms/step - accuracy: 0.5651 - loss: 0.6772 - val_accuracy: 0.5804 - val_loss: 0.6531
Epoch 2/100
1600/1600 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - accuracy: 0.5937 - loss: 0.6481 - val_accuracy: 0.6082 - val_loss: 0.6400
Epoch 3/100
1600/1600 ━━━━━━━━━━━━━━━━━━━━ 17s 11ms/step - accuracy: 0.6137 - loss: 0.6370 - val_accuracy: 0.6136 - val_loss: 0.6315
Epoch 4/100
1600/1600 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - accuracy: 0.6273 - loss: 0.6278 - val_accuracy: 0.6193 - val_loss: 0.6254
Epoch 5/100
1600/1600 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - accuracy: 0.6432 - loss: 0.6189 - val_accuracy: 0.6481 - val_loss: 0.6136
Epoch 6/100
1600/1600 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - accuracy: 0.6519 - loss: 0.6104 - val_accuracy: 0.6504 - val_loss: 0.6048
Epoch 7/100
1600/1600 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - accuracy: 0.6621 - loss: 0.6027 - val_accuracy: 0.6664 - val_loss: 0.6001
Epoch 8/100
1600/1600 ━━━━━━━━━━━━━━━━━━━━ 15s 10ms/step - accuracy: 

FINAL F1: 0.52608237304155
MAE: 0.3166833519935608
{'F1': 0.52608237304155, 'MAE': 0.3166833519935608}


In [ ]:
# Load the final results from the JSON file
with open("/content/drive/MyDrive/FYP/results.json", "r") as f:
    results = json.load(f)

print("\n" + "="*35)
print("     FINAL CLASSIFIER RESULTS     ")
print("="*35)
# Using formatted headers to match the data spacing
print(f"{'Appliance':<12} | {'F1':<8} | {'MAE':<8}")
print("-" * 35)

for app, r in results.items():
    # .4f ensures 4 decimal places for consistency
    print(f"{app.capitalize():<12} | {r['F1']:<8.4f} | {r['MAE']:<8.4f}")

print("="*35)


     FINAL CLASSIFIER RESULTS     
Appliance    | F1       | MAE     
-----------------------------------
Toaster      | 0.0000   | 0.2781  
Kettle       | 0.7500   | 0.0037  
Computer     | 0.6597   | 0.4661  
Lamp         | 0.5417   | 0.4080  


In [ ]:
model_filename = f"{MODEL_PATH}/{APP}_classifier.h5"
model.save(model_filename)
print(f"Model saved to: {model_filename}")

Model saved to: /content/drive/MyDrive/FYP/models/lamp_classifier.h5


In [ ]:
    drive_path = f"/content/drive/MyDrive/FYP/models/{app}_classifier.h5"

    if os.path.exists(drive_path):
        # 2. Trigger the browser download to your PC
        print(f"Downloading {app} model to your PC...")
        files.download(drive_path)
    else:
        print(f"Error: {app} model not found at {drive_path}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Convert to TensorFLow Lite

In [ ]:
import tensorflow as tf
import os

MODEL_PATH = "/content/drive/MyDrive/FYP/models"

APPLIANCES = ["kettle", "toaster", "computer", "lamp"]

for APP in APPLIANCES:

    print(f"\n===== Converting: {APP} =====")

    h5_path = f"{MODEL_PATH}/{APP}_classifier.h5"
    tflite_path = f"{MODEL_PATH}/{APP}_classifier.tflite"

    if not os.path.exists(h5_path):
        print(f"Model not found: {h5_path}")
        continue

    # load model
    model = tf.keras.models.load_model(h5_path)

    # convert
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]

    tflite_model = converter.convert()

    # save
    with open(tflite_path, "wb") as f:
        f.write(tflite_model)

    print(f"Saved: {tflite_path}")

print("\nAll conversions done!")


===== Converting: kettle =====


Saved artifact at '/tmp/tmpmb70j4x7'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 599, 1), dtype=tf.float32, name='input_layer_3')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  134525731125968: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134525525334096: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134525525338320: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134525525338128: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134525525336784: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134525525335632: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134525525337936: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134525525339088: TensorSpec(shape=(), dtype=tf.resource, name=None)
Saved: /content/drive/MyDrive/FYP/models/kettle_classifier.tflite

===== Converting: toaster =====


Saved artifact at '/tmp/tmp2uh88uhz'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 599, 2), dtype=tf.float32, name='input_layer_4')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  134525525342736: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134525525343696: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134525525333712: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134525525339472: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134525525338896: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134525525341200: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134525525339664: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134525525341968: TensorSpec(shape=(), dtype=tf.resource, name=None)
Saved: /content/drive/MyDrive/FYP/models/toaster_classifier.tflite

===== Converting: computer =====


Saved artifact at '/tmp/tmps7yb6r6w'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 599, 2), dtype=tf.float32, name='input_layer_5')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  134525494415760: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134525494421328: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134525494420944: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134525494422288: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134525494422480: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134525494421136: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134525494421712: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134525494420368: TensorSpec(shape=(), dtype=tf.resource, name=None)
Saved: /content/drive/MyDrive/FYP/models/computer_classifier.tflite

===== Converting: lamp =====


Saved artifact at '/tmp/tmpqbif7ike'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 599, 2), dtype=tf.float32, name='input_layer_6')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  134525494426704: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134525494427088: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134525494426896: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134525494427280: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134525494425936: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134525494426128: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134525494426512: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134525494424784: TensorSpec(shape=(), dtype=tf.resource, name=None)
Saved: /content/drive/MyDrive/FYP/models/lamp_classifier.tflite

All conversions done!
